In [10]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

In [11]:
def preprocess_any_csv(df):
    df = df.copy()

    # Drop fully empty columns
    df = df.dropna(axis=1, how="all")

    # Drop common label columns if present
    possible_labels = ["label", "Label", "class", "Class", "target", "Target", "attack", "Attack"]
    label_cols = [col for col in possible_labels if col in df.columns]
    if label_cols:
        df = df.drop(columns=label_cols)

    # Fill missing values
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].fillna("Unknown")
        else:
            df[col] = df[col].fillna(df[col].median())

    # Convert categorical columns using one-hot encoding
    cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
    if cat_cols:
        df = pd.get_dummies(df, columns=cat_cols)

    # Keep only numeric data
    df = df.select_dtypes(include=[np.number])

    return df

In [12]:
def build_autoencoder(input_dim):
    input_layer = Input(shape=(input_dim,))

    encoded = Dense(32, activation="relu")(input_layer)
    encoded = Dense(16, activation="relu")(encoded)
    encoded = Dense(8, activation="relu")(encoded)

    decoded = Dense(16, activation="relu")(encoded)
    decoded = Dense(32, activation="relu")(decoded)
    decoded = Dense(input_dim, activation="sigmoid")(decoded)

    autoencoder = Model(inputs=input_layer, outputs=decoded)
    autoencoder.compile(optimizer="adam", loss="mse")

    return autoencoder

In [13]:
def analyze_any_csv(df_raw):
    # Step 1: preprocess
    df_processed = preprocess_any_csv(df_raw)

    if df_processed.shape[1] < 2:
        raise ValueError("Not enough usable numeric/categorical columns to analyze.")

    # Step 2: scale
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(df_processed)

    # Step 3: split
    X_train, X_val = train_test_split(X_scaled, test_size=0.2, random_state=42)

    # Step 4: build model
    autoencoder = build_autoencoder(X_train.shape[1])

    # Step 5: train
    autoencoder.fit(
        X_train, X_train,
        epochs=10,
        batch_size=64,
        validation_data=(X_val, X_val),
        verbose=0
    )

    # Step 6: validation threshold
    X_val_pred = autoencoder.predict(X_val, verbose=0)
    val_mse = np.mean(np.power(X_val - X_val_pred, 2), axis=1)
    threshold = np.mean(val_mse) + 3 * np.std(val_mse)

    # Step 7: analyze all rows
    X_pred = autoencoder.predict(X_scaled, verbose=0)
    errors = np.mean(np.power(X_scaled - X_pred, 2), axis=1)
    predictions = (errors >= threshold).astype(int)

    severity = []
    actions = []

    for e in errors:
        if e < threshold:
            severity.append("Low")
            actions.append("Allow")
        elif e < 2 * threshold:
            severity.append("Medium")
            actions.append("Rate Limit")
        else:
            severity.append("High")
            actions.append("Block")

    results = df_raw.copy()
    results["Anomaly_Score"] = errors
    results["Prediction"] = predictions
    results["Severity"] = severity
    results["Mitigation_Action"] = actions

    # Optional network factor analysis if network-like columns exist
    if "src_bytes" in df_raw.columns and "dst_bytes" in df_raw.columns:
        src = pd.to_numeric(df_raw["src_bytes"], errors="coerce").fillna(0)
        dst = pd.to_numeric(df_raw["dst_bytes"], errors="coerce").fillna(0)
        results["Bandwidth_Stress"] = src + dst
    else:
        results["Bandwidth_Stress"] = errors * 100

    if "count" in df_raw.columns and "srv_count" in df_raw.columns:
        count = pd.to_numeric(df_raw["count"], errors="coerce").fillna(0)
        srv_count = pd.to_numeric(df_raw["srv_count"], errors="coerce").fillna(0)
        results["Traffic_Load"] = count + srv_count
    else:
        results["Traffic_Load"] = np.arange(1, len(results) + 1)

    results["Latency_Stress"] = results["Anomaly_Score"] * results["Traffic_Load"]
    results["Packet_Loss_Risk"] = results["Anomaly_Score"] * results["Bandwidth_Stress"]
    results["Throughput_Impact"] = 1 / (1 + results["Anomaly_Score"] + results["Traffic_Load"])

    results["Impact_Score"] = (
        0.3 * results["Bandwidth_Stress"] +
        0.3 * results["Latency_Stress"] +
        0.2 * results["Packet_Loss_Risk"] -
        0.2 * results["Throughput_Impact"]
    )

    return results, threshold

In [14]:
title = widgets.HTML(
    value="""
    <h2 style='color:#1f4e79;'>Generalized Anomaly Detection Dashboard</h2>
    <p>Upload any CSV file. The dashboard will preprocess it automatically, train an autoencoder, and detect anomalies.</p>
    """
)

uploader = widgets.FileUpload(
    accept=".csv",
    multiple=False,
    description="Upload CSV"
)

analyze_btn = widgets.Button(
    description="Analyze File",
    button_style="success"
)

output = widgets.Output(layout={"border": "1px solid #ccc", "padding": "10px"})

In [15]:
def run_dashboard(_):
    with output:
        clear_output(wait=True)

        if len(uploader.value) == 0:
            print("Please upload a CSV file.")
            return

        try:
            uploaded = uploader.value[0]
            raw_bytes = uploaded["content"].tobytes() if hasattr(uploaded["content"], "tobytes") else uploaded["content"]

            df_uploaded = pd.read_csv(io.BytesIO(raw_bytes))

            print("Uploaded file shape:", df_uploaded.shape)
            print("Columns:", list(df_uploaded.columns))

            results, threshold = analyze_any_csv(df_uploaded)

            display(widgets.HTML(f"<h3>Analysis Complete</h3><p><b>Threshold:</b> {threshold:.6f}</p>"))

            summary = pd.DataFrame({
                "Total Rows": [len(results)],
                "Attack Predictions": [(results["Prediction"] == 1).sum()],
                "Normal Predictions": [(results["Prediction"] == 0).sum()],
                "Average Anomaly Score": [results["Anomaly_Score"].mean()]
            })
            display(summary)

            display(results.head(20))

            plt.figure(figsize=(8,4))
            plt.plot(results["Anomaly_Score"].values, marker="o")
            plt.axhline(threshold, linestyle="--")
            plt.title("Anomaly Score")
            plt.xlabel("Row Index")
            plt.ylabel("Score")
            plt.show()

            plt.figure(figsize=(6,4))
            results["Mitigation_Action"].value_counts().plot(kind="bar")
            plt.title("Mitigation Actions")
            plt.xlabel("Action")
            plt.ylabel("Count")
            plt.show()

        except Exception as e:
            print("Error:", str(e))

In [16]:
analyze_btn.on_click(run_dashboard)

display(
    widgets.VBox([
        title,
        widgets.HBox([uploader, analyze_btn]),
        output
    ])
)